# CS604 Project - 01 Throughput test

Goal: find out how fast we can train ViT-B/16 on this GPU, and whether the **GPU** or the **data loading** is the bottleneck.
This gives the number of epochs that fit in the compute budget (2 h x 48 GB = 96 GB-hours), and the batch size to use.

Run on the same Colab server as notebook 00 (or just run the download cell again - it skips if the data is there).
Nothing here trains for real: the GPU test uses random tensors, the loader test only reads images.

## 0. Data (skipped if already prepared)

In [ ]:
%%bash
set -e
set -o pipefail
D=/content/cs604-course-data
U=https://huggingface.co/datasets/doem1997/cs604-course-data/resolve/main
if [ -f $D/.ready ]; then echo "Data already prepared - skipping download."; exit 0; fi
mkdir -p $D && cd $D
curl -sSfL $U/train.txt -o train.txt
curl -sSfL $U/val.txt -o val.txt
# stream both archives in parallel, extracting as they download
curl -sSfL $U/val.tar.gz   | tar -xz &
P1=$!
curl -sSfL $U/train.tar.gz | tar -xz
wait $P1
touch .ready
df -h /content
ls $D

## 1. Setup

In [ ]:
!pip install -q timm

In [ ]:
import os, time, random, statistics
import torch, timm
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader

ROOT = "/content/cs604-course-data"
MODEL = "vit_base_patch16_224.augreg2_in21k_ft_in1k"
NUM_CLASSES = 10000
N_TRAIN = 450_000
dev = "cuda"

props = torch.cuda.get_device_properties(0)
gib = props.total_memory / 2**30
GPU_GB = min([16, 24, 40, 48, 80], key=lambda v: abs(v - gib))   # nominal size
BUDGET_H = 96 / GPU_GB                                            # 2 h x 48 GB
print(f"{props.name}: {gib:.1f} GiB (nominal {GPU_GB} GB) -> budget {BUDGET_H:.2f} h per run")
print("CPU cores:", os.cpu_count(), "| compute capability:", torch.cuda.get_device_capability(0), "(bf16 is native from 8.0 = A100/L4; T4 is 7.5)")

def read_labels(name):
    with open(f"{ROOT}/{name}") as f:
        return [(p, int(c)) for p, c in (l.split() for l in f)]
train = read_labels("train.txt")

# new 10,000-class head on top of the pre-trained backbone
model = timm.create_model(MODEL, pretrained=True, num_classes=NUM_CLASSES).to(dev)
cfg = timm.data.resolve_data_config({}, model=model)
print("data config:", cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.1f} M")

## 2. Data loader speed (no GPU work)

Reads real training images with the training augmentation. Different random images per test so the disk cache does not flatter later tests.
`draft` = let the JPEG decoder shrink big images while decoding (faster if the images are much larger than 224 px).

In [ ]:
tf = T.Compose([
    T.RandomResizedCrop(224), T.RandomHorizontalFlip(), T.ToTensor(),
    T.Normalize(cfg["mean"], cfg["std"]),
])

class DS(Dataset):
    def __init__(self, rows, draft=False): self.rows, self.draft = rows, draft
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        p, c = self.rows[i]
        im = Image.open(f"{ROOT}/{p}")
        if self.draft: im.draft("RGB", (256, 256))
        return tf(im.convert("RGB")), c

def bench_loader(workers, draft, seed, n_batches=40, bs=128):
    rows = random.Random(seed).sample(train, bs * (n_batches + 5))
    dl = DataLoader(DS(rows, draft), batch_size=bs, num_workers=workers, pin_memory=True, prefetch_factor=4)
    it = iter(dl)
    for _ in range(5): next(it)                     # warm-up
    t, n = time.time(), 0
    for _ in range(n_batches):
        x, _ = next(it); n += len(x)
    ips = n / (time.time() - t)
    del it, dl
    return ips

ncpu = os.cpu_count()
loader = {}
for k, w in enumerate(sorted({w for w in (2, 4, 8, ncpu) if w <= ncpu})):
    loader[(w, False)] = bench_loader(w, False, seed=k)
    print(f"workers={w:>2}  draft=False  {loader[(w, False)]:7.0f} img/s")
loader[(ncpu, True)] = bench_loader(ncpu, True, seed=99)
print(f"workers={ncpu:>2}  draft=True   {loader[(ncpu, True)]:7.0f} img/s")
best_loader = max(loader.values())
print("best loader:", f"{best_loader:.0f} img/s")

## 3. GPU speed (random tensors, no data loading)

Full fine-tuning step: forward + backward + AdamW with mixed precision. Tests **bf16 and fp16** (a T4 has no native bf16, so fp16 is the one to use there;
L4/A100/H100 support bf16). Bigger batch is usually faster until memory runs out.

In [ ]:
def bench_train(bs, dtype, steps=12, warm=4):
    model.train(); model.requires_grad_(True)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5, fused=True)
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16))   # fp16 needs loss scaling
    x = torch.randn(bs, 3, 224, 224, device=dev); y = torch.randint(0, NUM_CLASSES, (bs,), device=dev)
    for i in range(warm + steps):
        if i == warm: torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=dtype):
            loss = F.cross_entropy(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    ips, peak = bs * steps / (time.time() - t), torch.cuda.max_memory_allocated() / 2**30
    del opt, x, y, loss
    return ips, peak

gpu = {}
for dname, dtype in (("bf16", torch.bfloat16), ("fp16", torch.float16)):
    for bs in (32, 64, 128, 256):
        try:
            gpu[(dname, bs)] = bench_train(bs, dtype)
            print(f"{dname} batch {bs:>3}: {gpu[(dname, bs)][0]:6.0f} img/s   peak VRAM {gpu[(dname, bs)][1]:5.1f} GiB")
        except torch.cuda.OutOfMemoryError:
            print(f"{dname} batch {bs:>3}: out of memory"); torch.cuda.empty_cache(); break
best_key = max(gpu, key=lambda k: gpu[k][0])
best_dname, best_bs = best_key
best_dtype = torch.bfloat16 if best_dname == "bf16" else torch.float16
best_gpu = gpu[best_key][0]
print(f"best GPU training speed: {best_gpu:.0f} img/s with {best_dname}, batch {best_bs}")

## 4. Inference speed and latency

Uses the fastest precision from section 3. The report needs inference latency; throughput also tells us the cost of one pass over the data (head initialisation, validation).

In [ ]:
model.eval()
@torch.inference_mode()
def bench_infer(bs, steps=20, warm=5):
    x = torch.randn(bs, 3, 224, 224, device=dev)
    for i in range(warm + steps):
        if i == warm: torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=best_dtype): model(x)
    torch.cuda.synchronize()
    return bs * steps / (time.time() - t)

infer_ips = bench_infer(256)
print(f"inference throughput (batch 256, {best_dname}): {infer_ips:.0f} img/s")

@torch.inference_mode()
def latency_ms(n=50):
    x = torch.randn(1, 3, 224, 224, device=dev); ts = []
    for i in range(n + 10):
        torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=best_dtype): model(x)
        torch.cuda.synchronize()
        if i >= 10: ts.append((time.time() - t) * 1000)
    return statistics.median(ts)
print(f"inference latency (batch 1, median): {latency_ms():.1f} ms")

## 5. Summary: what fits in the budget?

In [ ]:
train_ips = min(best_loader, best_gpu)
print(f"GPU: {props.name}")
print(f"loader {best_loader:.0f} img/s | GPU {best_gpu:.0f} img/s  ->  bottleneck: {'DATA LOADING' if best_loader < best_gpu else 'GPU'}")
epoch_min = N_TRAIN / train_ips / 60
print(f"one epoch over 450k images: {epoch_min:.1f} min")
print(f"epochs in {BUDGET_H:.2f} h: {BUDGET_H * 60 / epoch_min:.1f}   (before subtracting validation / head-init time)")
print(f"val pass (50k images, inference): {50_000 / infer_ips / 60:.1f} min, if the loader keeps up")
print(f"use {best_dname}, batch size {best_bs} (peak {gpu[best_key][1]:.1f} GiB)")